# Notebook 10: Mobile Benchmark & ONNX Runtime Integration Test

**Phase:** Deployment (Thesis Plan Deliverable 4)

**Purpose:** Validate ONNX-exported models for target mobile constraints: latency, model size, accuracy consistency against PyTorch baseline.

**Note:** The Android deployment target is ONNX Runtime Mobile (not TFLite). ONNX models exported by Notebook 09 are validated here against thesis acceptance criteria.

## Overview

This notebook validates that exported ONNX models meet the thesis success criteria:
- **Latency:** < 1 second per inference on mobile CPU (ONNX Runtime)
- **Size:** < 40 MB per model, < 120 MB combined RAM
- **Accuracy:** ONNX output matches PyTorch baseline within tolerance (5e-3)
- **Offline:** Models work without network connectivity

## Workflow

1. Load the deployment report from Notebook 09
2. Validate ONNX output consistency against PyTorch baseline
3. Measure model sizes
4. Benchmark latency on CPU (simulating mobile constraints)
5. Measure memory usage during inference
6. Run sample predictions with end-to-end pipeline
7. Generate mobile readiness report

In [1]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.data_utils import get_data_directories
from utils.text_processing import get_allergen_list
from utils.model_utils import load_model_and_tokenizer
from utils.deployment_utils import (
    validate_onnx_output,
    measure_model_size,
    get_model_size_report,
    benchmark_onnx_latency,
    print_deployment_status,
)

print("Imports successful.")

Imports successful.


## 1. Check Export Status

Verify that ONNX models were exported in Notebook 09 and load the deployment report.

In [2]:
dirs = get_data_directories()
export_dir = os.path.join(dirs['base'], 'models', 'exported')

# Load deployment report
report_path = os.path.join(export_dir, 'deployment_report.json')
if os.path.exists(report_path):
    report = json.load(open(report_path))
    print(f"Deployment report found. Date: {report.get('deployment_date', 'unknown')}")
    print(f"Deployment target: {report.get('deployment_target', 'unknown')}")
else:
    print("❌ No deployment report found — run Notebook 09 first.")
    report = {}

# Check what export files exist
print("\nExport directory contents:")
print("=" * 40)
if os.path.exists(export_dir):
    for root, dirs_f, files in os.walk(export_dir):
        level = root.replace(export_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = ' ' * 2 * (level + 1)
        for f in files:
            size = os.path.getsize(os.path.join(root, f))
            size_mb = size / (1024 * 1024)
            print(f"{sub_indent}{f}  ({size_mb:.1f} MB)")
else:
    print("  (empty — run Notebook 09 first)")

Deployment report found. Date: 2026-06-26T16:47:06.068393
Deployment target: ONNX Runtime Mobile (Android)

Export directory contents:
exported/
  deployment_report.json  (0.0 MB)
  mobile_readiness_report.json  (0.0 MB)
  allergen_model/
    allergen_model_export_config.json  (0.0 MB)
    allergen_model.onnx  (94.5 MB)


## 2. Thesis Success Criteria Check

Evaluate all success criteria from the thesis plan.
The deployment target is ONNX Runtime Mobile (Android), so model size and latency criteria apply to ONNX models.

In [3]:
# Check model sizes from export directory
onnx_path = os.path.join(export_dir, "allergen_model", "allergen_model.onnx")
onnx_model_size = measure_model_size(onnx_path) if os.path.exists(onnx_path) else {"size_mb": 0}

criteria = [
    {"metric": "ONNX Model Size", "target": "< 40 MB", 
     "status": f"{'✅' if onnx_model_size['size_mb'] < 40 else '❌'} {onnx_model_size['size_mb']:.1f} MB"},
    {"metric": "Combined RAM", "target": "< 120 MB", "status": "⚠️ Not yet measured on device"},
    {"metric": "Mobile Inference Latency", "target": "< 1 second", 
     "status": f"{'⚠️' if onnx_model_size['size_mb'] > 0 else '❌'} Desktop CPU benchmark below — mobile TBD"},
    {"metric": "Offline Operation", "target": "Yes", "status": "✅ ONNX model runs offline (no network needed)"},
    {"metric": "ONNX Output Validated", "target": "Matches PyTorch (> 5e-3 tolerance)",
     "status": report.get("allergen_model", {}).get("onnx_validated", False) and "✅ Validated" or "⚠️ Not yet validated"},
]

criteria_df = pd.DataFrame(criteria)
print("Mobile Success Criteria Status:")
print("=" * 60)
for _, row in criteria_df.iterrows():
    print(f"  {row['metric']:35s}  target={row['target']:25s}  {row['status']}")

Mobile Success Criteria Status:
  ONNX Model Size                      target=< 40 MB                    ❌ 94.5 MB
  Combined RAM                         target=< 120 MB                   ⚠️ Not yet measured on device
  Mobile Inference Latency             target=< 1 second                 ⚠️ Desktop CPU benchmark below — mobile TBD
  Offline Operation                    target=Yes                        ✅ ONNX model runs offline (no network needed)
  ONNX Output Validated                target=Matches PyTorch (> 5e-3 tolerance)  ✅ Validated


## 3. ONNX Validation (Desktop)

Compare ONNX Runtime output against PyTorch baseline to check for accuracy degradation.
Tolerance set at 5e-3 per thesis plan.

In [4]:
# Load PyTorch model for ONNX validation
allergen_model_path = "../models/mobilebert_allergen_final/"
if not os.path.exists(onnx_path):
    print("❌ ONNX model not found. Run Notebook 09 first.")
elif not os.path.exists(allergen_model_path):
    print("❌ PyTorch model not found.")
else:
    # Load PyTorch model
    print("Loading PyTorch model for baseline comparison...")
    model, tokenizer, device = load_model_and_tokenizer(allergen_model_path)
    
    print("\nONNX Validation (vs PyTorch baseline):")
    print("=" * 50)
    
    # Run ONNX validation on a representative test sample
    val = validate_onnx_output(
        onnx_path=onnx_path,
        pytorch_model=model,
        tokenizer=tokenizer,
        test_text="milk powder, sugar, cocoa butter, soy lecithin, salt, wheat flour, eggs, vanilla extract",
        tolerance=5e-3,
    )
    
    if val["within_tolerance"]:
        print(f"\n✅ ONNX model validated — max diff {val['max_diff']:.6e} is within tolerance ({val['tolerance']}).")
    else:
        print(f"\n❌ ONNX output exceeds tolerance ({val['max_diff']:.6e} > {val['tolerance']}).")
        print("   Fallback: Use PyTorch Mobile for deployment instead of ONNX Runtime.")
    
    # Also validate on common Philippine products
    print("\n\nValidation on multiple test samples:")
    test_samples = [
        "wheat flour, water, sugar, yeast, salt",
        "enriched flour, sugar, vegetable oil, corn syrup, leavening",
        "milk, sugar, cream, whey, stabilizer, natural flavors",
        "corn, palm oil, salt, sugar, msg, onion powder",
    ]
    all_pass = True
    for sample in test_samples:
        v = validate_onnx_output(onnx_path, model, tokenizer, sample, tolerance=5e-3)
        passed = "✅" if v["within_tolerance"] else "❌"
        print(f"  {passed} max_diff={v['max_diff']:.2e}  text='{sample[:50]}...'")
        all_pass = all_pass and v["within_tolerance"]
    
    print(f"\nOverall: {'✅ ALL PASS' if all_pass else '❌ SOME FAILED'}")
    if all_pass:
        print("✅ ONNX Runtime Mobile is a viable deployment target.")
        print("   Proceed with Android integration.")
    else:
        print("⚠️  ONNX accuracy degradation detected. Consider PyTorch Mobile as fallback.")

Loading PyTorch model for baseline comparison...

ONNX Validation (vs PyTorch baseline):
[deployment_utils] ONNX output validation
[deployment_utils]   Max diff vs PyTorch:  3.242493e-05
[deployment_utils]   Mean diff vs PyTorch: 1.639128e-05
[deployment_utils]   Within tolerance (0.005): ✅ YES

✅ ONNX model validated — max diff 3.242493e-05 is within tolerance (0.005).


Validation on multiple test samples:
[deployment_utils] ONNX output validation
[deployment_utils]   Max diff vs PyTorch:  1.525879e-05
[deployment_utils]   Mean diff vs PyTorch: 7.033348e-06
[deployment_utils]   Within tolerance (0.005): ✅ YES
  ✅ max_diff=1.53e-05  text='wheat flour, water, sugar, yeast, salt...'
[deployment_utils] ONNX output validation
[deployment_utils]   Max diff vs PyTorch:  2.670288e-05
[deployment_utils]   Mean diff vs PyTorch: 6.794930e-06
[deployment_utils]   Within tolerance (0.005): ✅ YES
  ✅ max_diff=2.67e-05  text='enriched flour, sugar, vegetable oil, corn syrup, ...'
[deployment_utils]

## 4. End-to-End Inference Test

Run a complete inference pipeline: ingredient text → ONNX prediction, simulating mobile usage.

In [5]:
# Load ONNX model for end-to-end testing
import onnxruntime as ort

if os.path.exists(onnx_path):
    ort_session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    
    # Test samples (common Philippine products)
    test_products = [
        ("SkyFlakes Crackers", "wheat flour, vegetable oil, sugar, salt, yeast"),
        ("Lucky Me! Noodles", "wheat flour, palm oil, salt, sugar, soy sauce, monosodium glutamate"),
        ("Magnolia Ice Cream", "milk, sugar, cream, whey, stabilizer, natural flavors"),
        ("Chippy", "corn, palm oil, salt, sugar, msg, onion powder"),
    ]
    
    print("End-to-end ONNX inference tests:")
    print("=" * 50)
    allergen_list = get_allergen_list()
    
    for name, ingredients in test_products:
        encoded = tokenizer([ingredients], padding=True, truncation=True, max_length=221, return_tensors="pt")
        ort_inputs = {
            "input_ids": encoded["input_ids"].numpy(),
            "attention_mask": encoded["attention_mask"].numpy(),
        }
        ort_logits = ort_session.run(["logits"], ort_inputs)[0]
        probs = 1 / (1 + np.exp(-ort_logits))  # sigmoid
        preds = (probs >= 0.5).astype(int)[0]
        detected = [allergen_list[i] for i, v in enumerate(preds) if v == 1]
        print(f"\n  Product: {name}")
        print(f"  Ingredients: {ingredients[:80]}...")
        print(f"  Detected allergens: {detected}")
else:
    print("ONNX model not available — run Notebook 09 first.")

End-to-end ONNX inference tests:

  Product: SkyFlakes Crackers
  Ingredients: wheat flour, vegetable oil, sugar, salt, yeast...
  Detected allergens: ['wheat']

  Product: Lucky Me! Noodles
  Ingredients: wheat flour, palm oil, salt, sugar, soy sauce, monosodium glutamate...
  Detected allergens: ['soy', 'wheat']

  Product: Magnolia Ice Cream
  Ingredients: milk, sugar, cream, whey, stabilizer, natural flavors...
  Detected allergens: ['milk']

  Product: Chippy
  Ingredients: corn, palm oil, salt, sugar, msg, onion powder...
  Detected allergens: []


## 5. Desktop Latency Benchmark

Measure inference latency on CPU to estimate mobile performance.
Mobile devices typically have 2-4× higher latency than desktop CPUs.

In [6]:
if os.path.exists(onnx_path) and os.path.exists(allergen_model_path):
    benchmark_texts = [
        "milk powder, sugar, cocoa butter, soy lecithin, salt",
        "wheat flour, water, sugar, yeast, salt",
        "enriched flour, sugar, vegetable oil, corn syrup, leavening",
        "corn syrup, sugar, palm oil, salt, whey, natural flavor",
    ]
    
    # --- ONNX Latency ---
    print("ONNX Inference Latency (CPU, desktop):")
    print("-" * 45)
    encoded = tokenizer(benchmark_texts, padding=True, truncation=True, max_length=221, return_tensors="pt")
    onnx_results = benchmark_onnx_latency(
        onnx_path=onnx_path,
        input_ids=encoded["input_ids"].numpy(),
        attention_mask=encoded["attention_mask"].numpy(),
        n_runs=100,
    )
    print(f"  Mean:   {onnx_results['mean_ms']:.1f} ms")
    print(f"  Median: {onnx_results['median_ms']:.1f} ms")
    print(f"  P95:    {onnx_results['p95_ms']:.1f} ms")
    print(f"  Std:    {onnx_results['std_ms']:.1f} ms")
    print(f"  Runs:   {onnx_results['runs']}")
    
    # --- Estimate mobile latency ---
    mobile_multiplier = 3.0  # conservative estimate: mobile CPU 3× slower than desktop
    estimated_mobile_ms = onnx_results['mean_ms'] * mobile_multiplier
    print(f"\n📱 Estimated mobile latency ({mobile_multiplier}x desktop): {estimated_mobile_ms:.0f} ms")
    print(f"   Thesis target: < 1000 ms")
    print(f"   {'✅ PASS' if estimated_mobile_ms < 1000 else '❌ FAIL'}")
else:
    print("Skipping benchmarks — models not available. Run Notebook 09 first.")

ONNX Inference Latency (CPU, desktop):
---------------------------------------------
  Mean:   14.9 ms
  Median: 15.0 ms
  P95:    15.4 ms
  Std:    0.3 ms
  Runs:   100

📱 Estimated mobile latency (3.0x desktop): 45 ms
   Thesis target: < 1000 ms
   ✅ PASS


## 6. Memory Usage During Inference

Measure approximate memory usage during ONNX inference to validate combined RAM criteria.

In [7]:
if os.path.exists(onnx_path):
    # Measure model file size (on-disk)
    model_size = measure_model_size(onnx_path)
    print("ONNX Model File Size:")
    print(f"  On disk: {model_size['size_mb']:.1f} MB")
    
    # Estimate runtime memory: ONNX Runtime loads into memory ≈ 1.5× file size
    estimated_ram = model_size['size_mb'] * 1.5
    print(f"  Estimated runtime RAM (1.5× file size): {estimated_ram:.1f} MB")
    
    # PyTorch model size for comparison
    pt_model_size = measure_model_size(os.path.join(allergen_model_path, "model.safetensors"))
    pt_filesize = pt_model_size.get('size_mb', 0)
    if pt_filesize > 0:
        print(f"\nPyTorch model file size: {pt_filesize:.1f} MB")
        print(f"  ONNX is {model_size['size_mb'] / pt_filesize:.1%} of PyTorch size")
    
    print(f"\nCombined storage estimate (allergen ONNX): {model_size['size_mb']:.1f} MB")
    print(f"✅ Thesis criteria: < 40 MB per model")
    if model_size['size_mb'] < 40:
        print(f"   PASS ({model_size['size_mb']:.1f} MB < 40 MB)")
    else:
        print(f"   ⚠️  {'PASS' if model_size['size_mb'] < 80 else 'FAIL'} — model is {'within combined target' if model_size['size_mb'] < 80 else 'over both per-model and combined targets'}")
else:
    print("Skipping memory measurement — model not available.")

ONNX Model File Size:
  On disk: 94.5 MB
  Estimated runtime RAM (1.5× file size): 141.7 MB

PyTorch model file size: 93.9 MB
  ONNX is 100.6% of PyTorch size

Combined storage estimate (allergen ONNX): 94.5 MB
✅ Thesis criteria: < 40 MB per model
   ⚠️  FAIL — model is over both per-model and combined targets


## 7. Mobile Readiness Report

Generate a final summary table of thesis success criteria status for documentation.

In [8]:
# Build mobile readiness report
readiness = {}

if os.path.exists(onnx_path):
    model_size = measure_model_size(onnx_path)
    readiness["model_size_mb"] = model_size["size_mb"]
    readiness["model_size_pass"] = model_size["size_mb"] < 40
    readiness["onnx_available"] = True
else:
    readiness["model_size_mb"] = None
    readiness["model_size_pass"] = False
    readiness["onnx_available"] = False

# Check ONNX validation from earlier cell using try/except
try:
    readiness["onnx_validated"] = all_pass
except NameError:
    readiness["onnx_validated"] = None

# Check latency results using try/except
try:
    l = onnx_results
    readiness["latency_mean_ms"] = l["mean_ms"]
    readiness["latency_p95_ms"] = l["p95_ms"]
    readiness["estimated_mobile_ms"] = l["mean_ms"] * 3.0
    readiness["latency_pass"] = (l["mean_ms"] * 3.0) < 1000
except NameError:
    readiness["latency_mean_ms"] = None
    readiness["latency_pass"] = False

# Print final report
print("\n" + "=" * 55)
print("    MOBILE READINESS REPORT — ONNX Runtime Mobile")
print("=" * 55 + "\n")

rows = [
    ("ONNX Model Available", "✅ Yes" if readiness["onnx_available"] else "❌ No"),
    ("Model Size", f"{readiness['model_size_mb']:.1f} MB{' ✅' if readiness['model_size_pass'] else ' ❌'} (target < 40 MB)"),
    ("ONNX Output Validated", f"{'✅ Yes' if readiness['onnx_validated'] else '⚠️ Not tested' if readiness['onnx_validated'] is None else '❌ Failed'}"),
]
if readiness["latency_mean_ms"] is not None:
    rows.append(("Desktop Latency (mean)", f"{readiness['latency_mean_ms']:.1f} ms"))
    rows.append(("Estimated Mobile Latency", f"{readiness['estimated_mobile_ms']:.0f} ms{' ✅' if readiness['latency_pass'] else ' ❌'} (target < 1000 ms)"))
rows.append(("Deployment Target", "ONNX Runtime Mobile (Android)"))
rows.append(("Offline Operation", "✅ Yes — no network required"))
rows.append(("Android Integration", "📋 Next step: Integrate into Android app via ONNX Runtime Mobile SDK"))

for metric, status in rows:
    print(f"  {metric:35s}  {status}")

print("\n" + "=" * 55)

# Save readiness report
readiness_path = os.path.join(export_dir, "mobile_readiness_report.json")
with open(readiness_path, "w") as f:
    json.dump(readiness, f, indent=2, default=str)
print(f"\nReadiness report saved to {readiness_path}")


    MOBILE READINESS REPORT — ONNX Runtime Mobile

  ONNX Model Available                 ✅ Yes
  Model Size                           94.5 MB ❌ (target < 40 MB)
  ONNX Output Validated                ✅ Yes
  Desktop Latency (mean)               14.9 ms
  Estimated Mobile Latency             45 ms ✅ (target < 1000 ms)
  Deployment Target                    ONNX Runtime Mobile (Android)
  Offline Operation                    ✅ Yes — no network required
  Android Integration                  📋 Next step: Integrate into Android app via ONNX Runtime Mobile SDK


Readiness report saved to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/exported/mobile_readiness_report.json
